# Projet A3 · Reconnaissance d'images · ⭐⭐⭐

**Question métier** : peut-on reconnaître automatiquement ce qu'il y a sur une photo (un vêtement, une fleur) assez bien pour s'en servir dans un vrai produit ?
**Ce qu'on va construire** : un CNN « maison » sur Fashion-MNIST, puis un modèle pré-entraîné (MobileNetV2) **transféré puis fine-tuné** sur des photos de fleurs, avec courbes, matrice de confusion et Grad-CAM pour voir ce que le modèle regarde.
**Livrable** : ce notebook complété + un rapport de 10-15 slides (gabarit dans `../gabarit-rapport.md`) + une phrase de synthèse.

**Comment l'utiliser**
- Google Colab (menu *Exécution → Modifier le type d'exécution → GPU T4*) ou en local avec `../requirements.txt`. `Maj + Entrée` pour exécuter une cellule.
- `MODE_RAPIDE = True` (par défaut) : sous-échantillons et 2 epochs, tout tourne en moins de 10 min même sans GPU. Mets `False` sur Colab GPU pour la version complète (≈ 8 min sur T4).
- Les cellules **« À toi »** sont des exercices : elles s'exécutent telles quelles, la vérification affiche ✅ ou ❌, la solution est cachée juste en dessous — essaie avant de l'ouvrir.
- Les cellules **« Rapport »** impriment les chiffres à recopier dans tes slides.

## 0. Préparation

Imports, flag `MODE_RAPIDE`, helpers de vérification et dossier `data/` (ignoré par git) pour les téléchargements.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"     # moins de messages TensorFlow
import io
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [ ]:
MODE_RAPIDE = True    # True : sous-échantillons + 2 epochs (< 10 min sur CPU). False : version complète (Colab GPU T4, ≈ 8 min)

EPOCHS = 2 if MODE_RAPIDE else 12             # pour chaque entraînement
N_FASHION = 10_000 if MODE_RAPIDE else 60_000  # images Fashion-MNIST utilisées (train)
N_TEST = 2_000 if MODE_RAPIDE else 10_000      # images Fashion-MNIST de test
PART_PHOTOS = 0.15 if MODE_RAPIDE else 1.0     # part des 3 670 photos de fleurs chargée
IMG_SIZE = 96 if MODE_RAPIDE else 160          # taille des photos pour MobileNetV2 (96, 128, 160, 192 ou 224)
GRAINE = 42

tf.random.set_seed(GRAINE)
np.random.seed(GRAINE)
DATA_DIR = os.path.abspath("data")
os.makedirs(DATA_DIR, exist_ok=True)
print("TensorFlow", tf.__version__, "· GPU :", tf.config.list_physical_devices("GPU") or "aucun", "· MODE_RAPIDE =", MODE_RAPIDE)

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans arrêter le notebook."""
    print(("✅ " if condition else "❌ ") + nom)

def proche(a, b, tolerance=1e-3):
    """Vrai si a et b sont égaux à `tolerance` près."""
    return abs(float(a) - float(b)) <= tolerance

def tracer_historique(historiques, titre=""):
    """Courbes loss / accuracy (train en trait plein, validation en pointillé) pour {nom: history}."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    for nom, h in historiques.items():
        ax1.plot(h.history["loss"], label=f"{nom} · train")
        ax1.plot(h.history["val_loss"], "--", label=f"{nom} · validation")
        ax2.plot(h.history["accuracy"], label=f"{nom} · train")
        ax2.plot(h.history["val_accuracy"], "--", label=f"{nom} · validation")
    ax1.set_title("Perte (loss)"); ax2.set_title("Accuracy")
    ax1.set_xlabel("epoch"); ax2.set_xlabel("epoch")
    ax1.legend(fontsize=8); ax2.legend(fontsize=8)
    fig.suptitle(titre); plt.tight_layout(); plt.show()

print("Helpers prêts.")

## 1. Contexte et question métier

Une boutique en ligne reçoit chaque jour des milliers de photos de produits envoyées par ses vendeurs. Les classer à la main (« est-ce un pull ou un manteau ? ») coûte du temps et produit des erreurs. Même besoin dans une application de jardinage qui veut nommer une fleur à partir d'une photo prise avec un téléphone. Dans les deux cas, la question est la même : **un modèle peut-il reconnaître la catégorie d'une image assez bien pour remplacer (ou aider) un humain ?**

On travaille en deux temps. D'abord **Fashion-MNIST** : 70 000 petites images de vêtements en noir et blanc, 10 catégories, le terrain idéal pour construire et comprendre un réseau convolutif (CNN) de A à Z. Ensuite **tf_flowers** : 3 670 vraies photos de fleurs, 5 espèces, couleurs, fonds et cadrages variés — là, un CNN maison ne suffit plus et on réutilise un modèle pré-entraîné sur ImageNet (**transfert**) que l'on adapte à nos fleurs (**fine-tuning**).

À qui ça sert : à l'équipe produit, qui doit décider si elle déploie l'étiquetage automatique, avec quel niveau de vérification humaine. Un bon résultat, c'est une **accuracy** élevée (les classes sont équilibrées, donc l'accuracy est une métrique honnête ici) **et** une matrice de confusion dont les erreurs sont « compréhensibles » (chemise ↔ T-shirt, pas robe ↔ basket). Repères : au-delà de 90 % sur Fashion-MNIST et de 85 % sur les fleurs, le modèle vaut un test en production avec relecture humaine des cas incertains.

## 2. Les données

### 2.1 Fashion-MNIST (Zalando)

Intégré à Keras : rien à télécharger à la main. 60 000 images d'apprentissage + 10 000 de test, 28×28 pixels en niveaux de gris, 10 classes équilibrées. Licence MIT.

| Variable | Contenu | Type |
|---|---|---|
| `X` | image 28×28, un entier par pixel (0 = noir, 255 = blanc) | `uint8`, forme `(N, 28, 28)` |
| `y` | code de la classe, de 0 à 9 (dictionnaire ci-dessous) | `uint8`, forme `(N,)` |

In [ ]:
(X_fm_train, y_fm_train), (X_fm_test, y_fm_test) = keras.datasets.fashion_mnist.load_data()
X_fm_train, y_fm_train = X_fm_train[:N_FASHION], y_fm_train[:N_FASHION]
X_fm_test, y_fm_test = X_fm_test[:N_TEST], y_fm_test[:N_TEST]

print("train :", X_fm_train.shape, "· test :", X_fm_test.shape, "· type :", X_fm_train.dtype,
      "· pixels de", X_fm_train.min(), "à", X_fm_train.max())
print("Valeurs manquantes :", int(np.isnan(X_fm_train.astype(float)).sum()),
      "· images entièrement noires :", int((X_fm_train.max(axis=(1, 2)) == 0).sum()))

In [ ]:
CLASSES_FM = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
CLASSES_FM_FR = ["T-shirt", "Pantalon", "Pull", "Robe", "Manteau", "Sandale", "Chemise", "Basket", "Sac", "Bottine"]

dictionnaire_fm = pd.DataFrame({"code": range(10), "classe (en)": CLASSES_FM, "classe (fr)": CLASSES_FM_FR,
                                "nb train": np.bincount(y_fm_train), "nb test": np.bincount(y_fm_test)})
dictionnaire_fm

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for code, ax in enumerate(axes.flat):
    idx = np.where(y_fm_train == code)[0][0]
    ax.imshow(X_fm_train[idx], cmap="gray")
    ax.set_title(f"{code} · {CLASSES_FM_FR[code]}"); ax.axis("off")
plt.suptitle("Une image par classe (Fashion-MNIST)"); plt.tight_layout(); plt.show()

### 2.2 tf_flowers (photos de fleurs)

3 670 photos JPEG de tailles variées, 5 classes : `daisy` (pâquerette), `dandelion` (pissenlit), `roses`, `sunflowers` (tournesols), `tulips`. C'est l'archive officielle du dataset `tf_flowers` de TensorFlow Datasets (218 Mo, licence CC-BY, crédits photo dans `LICENSE.txt`) ; on la télécharge directement dans `data/` avec `keras.utils.get_file` (mise en cache : le second lancement ne retélécharge rien).

**Repli** : si le téléchargement échoue (pas de réseau), le notebook continue avec Fashion-MNIST pour toute la partie « photos » — le code est le même, seuls les résultats changent.

In [ ]:
URL_FLEURS = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

def lister_photos_fleurs():
    """Télécharge (une fois) et liste les photos : (classes, chemins, codes)."""
    keras.utils.get_file("flower_photos.tgz", URL_FLEURS, extract=True, cache_dir=DATA_DIR, cache_subdir=".")
    racine = next(os.path.join(r, "flower_photos") for r, dossiers, _ in os.walk(DATA_DIR) if "flower_photos" in dossiers)
    classes = sorted(d for d in os.listdir(racine) if os.path.isdir(os.path.join(racine, d)))
    chemins, codes = [], []
    for code, classe in enumerate(classes):
        fichiers = sorted(f for f in os.listdir(os.path.join(racine, classe)) if f.endswith(".jpg"))
        chemins += [os.path.join(racine, classe, f) for f in fichiers]
        codes += [code] * len(fichiers)
    return classes, np.array(chemins), np.array(codes)

try:
    CLASSES_PHOTOS, chemins_photos, codes_photos = lister_photos_fleurs()
    SOURCE_PHOTOS = "tf_flowers"
    print(f"tf_flowers : {len(chemins_photos)} photos · classes : {CLASSES_PHOTOS}")
except Exception as e:
    print("Téléchargement impossible (", type(e).__name__, ") → repli : Fashion-MNIST servira aussi de jeu « photos ».")
    SOURCE_PHOTOS = "Fashion-MNIST (repli)"
    CLASSES_PHOTOS, chemins_photos, codes_photos = CLASSES_FM_FR, None, None

In [ ]:
if SOURCE_PHOTOS == "tf_flowers":
    dictionnaire_photos = pd.DataFrame({"code": range(len(CLASSES_PHOTOS)), "classe": CLASSES_PHOTOS,
                                        "nb photos": np.bincount(codes_photos)})
    print(dictionnaire_photos.to_string(index=False))
    fig, axes = plt.subplots(1, 5, figsize=(13, 3))
    for code, ax in enumerate(axes):
        image = plt.imread(chemins_photos[codes_photos == code][0])
        ax.imshow(image); ax.set_title(f"{CLASSES_PHOTOS[code]} · {image.shape[1]}×{image.shape[0]} px"); ax.axis("off")
    plt.suptitle("Une photo par classe, à leur taille d'origine (variable !)"); plt.tight_layout(); plt.show()
else:
    print("Repli actif : le dictionnaire des classes est celui de Fashion-MNIST (voir plus haut).")

## 3. Nettoyage et feature engineering

Pour des images, « nettoyer » veut dire : mettre tous les pixels sur la même échelle, donner à chaque image la **même forme** (taille, nombre de canaux), séparer un jeu de **validation** pour surveiller le sur-apprentissage, et fabriquer des variantes des images (**augmentation**) pour que le modèle ne les apprenne pas par cœur. Chaque décision est notée dans un `JOURNAL` affiché en fin de section.

In [ ]:
JOURNAL = []

def noter(etape, detail, effet):
    JOURNAL.append({"étape": etape, "détail": detail, "effet": effet})

**À toi · exercice 1 · normaliser les pixels**

Les pixels vont de 0 à 255 ; un réseau apprend mieux avec des entrées entre 0 et 1. Crée `X_train_n` et `X_test_n` en `float32` entre 0 et 1, avec un axe « canal » en plus pour Keras : forme `(N, 28, 28, 1)`.

<details><summary>Indice</summary>

`(X / 255.0).astype("float32")[..., np.newaxis]` ajoute une dimension à la fin.
</details>

In [ ]:
# À toi : pixels / 255 en float32, puis ajoute l'axe du canal → forme (N, 28, 28, 1)
X_train_n = None
X_test_n = None

In [ ]:
verifier("Exercice 1 · pixels entre 0 et 1 en float32", X_train_n is not None and float(X_train_n.max()) <= 1.0 and X_train_n.dtype == np.float32)
verifier("Exercice 1 · forme (N, 28, 28, 1)", X_train_n is not None and X_train_n.shape[1:] == (28, 28, 1))
if X_train_n is None:      # filet de sécurité : le notebook continue même sans ta réponse
    X_train_n = (X_fm_train / 255.0).astype("float32")[..., np.newaxis]
    X_test_n = (X_fm_test / 255.0).astype("float32")[..., np.newaxis]
noter("Normalisation", "pixels / 255 (float32) + axe canal", "entrées entre 0 et 1, forme (N, 28, 28, 1)")

<details><summary>Solution de l'exercice 1</summary>

```python
X_train_n = (X_fm_train / 255.0).astype("float32")[..., np.newaxis]
X_test_n = (X_fm_test / 255.0).astype("float32")[..., np.newaxis]
```
</details>

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train_n, y_fm_train, test_size=0.2, stratify=y_fm_train, random_state=GRAINE)
noter("Découpage", "20 % du train gardés en validation (stratifié)", f"{len(X_tr)} apprentissage / {len(X_val)} validation / {len(X_test_n)} test")
print(len(X_tr), "images d'apprentissage ·", len(X_val), "de validation ·", len(X_test_n), "de test")

Pour les photos, on ramène tout à `IMG_SIZE × IMG_SIZE × 3` et on garde les pixels **entre 0 et 255** : la mise à l'échelle attendue par MobileNetV2 (entre −1 et 1) sera faite *dans* le modèle par une couche `Rescaling`. Avantage : une photo téléchargée en section 7 n'aura besoin d'aucun prétraitement.

In [ ]:
def charger_photo(chemin, taille=IMG_SIZE):
    image = tf.io.decode_jpeg(tf.io.read_file(chemin), channels=3)
    return tf.image.resize(image, (taille, taille)).numpy()

if SOURCE_PHOTOS == "tf_flowers":
    if PART_PHOTOS < 1:
        chemins_photos, _, codes_photos, _ = train_test_split(chemins_photos, codes_photos, train_size=PART_PHOTOS,
                                                              stratify=codes_photos, random_state=GRAINE)
    X_photos = np.stack([charger_photo(c) for c in chemins_photos]).astype("float32")
    y_photos = codes_photos
else:                                         # repli : Fashion-MNIST converti en « photos » RVB
    idx = np.random.RandomState(GRAINE).choice(len(X_fm_train), size=int(3670 * PART_PHOTOS), replace=False)
    X_photos = tf.image.resize(tf.image.grayscale_to_rgb(tf.constant(X_fm_train[idx][..., np.newaxis], tf.float32)),
                               (IMG_SIZE, IMG_SIZE)).numpy()
    y_photos = y_fm_train[idx]
noter("Redimensionnement", f"photos ramenées à {IMG_SIZE}×{IMG_SIZE}, 3 canaux, pixels 0-255", f"tenseur unique {X_photos.shape}")
print(SOURCE_PHOTOS, ":", X_photos.shape, X_photos.dtype)

In [ ]:
Xp_tr, Xp_test, yp_tr, yp_test = train_test_split(X_photos, y_photos, test_size=0.2, stratify=y_photos, random_state=GRAINE)
Xp_tr, Xp_val, yp_tr, yp_val = train_test_split(Xp_tr, yp_tr, test_size=0.2, stratify=yp_tr, random_state=GRAINE)
noter("Découpage photos", "64 % / 16 % / 20 % (stratifié)", f"{len(Xp_tr)} apprentissage / {len(Xp_val)} validation / {len(Xp_test)} test")
print(len(Xp_tr), "photos d'apprentissage ·", len(Xp_val), "de validation ·", len(Xp_test), "de test")

**À toi · exercice 2 · une recette d'augmentation**

L'augmentation crée à la volée des variantes de chaque image (miroir, rotation, zoom) pendant l'entraînement. Écris `faire_augmentation()` qui renvoie un **nouveau** `models.Sequential` de 3 couches : `layers.RandomFlip("horizontal")`, `layers.RandomRotation(0.1)`, `layers.RandomZoom(0.1)`. (Une fonction plutôt qu'un objet : chaque modèle aura sa propre copie.)

<details><summary>Indice</summary>

`models.Sequential([couche1, couche2, couche3])` — les trois couches sont dans `layers`.
</details>

In [ ]:
# À toi : renvoie un Sequential avec les 3 couches d'augmentation
def faire_augmentation():
    return None

In [ ]:
essai = faire_augmentation()
verifier("Exercice 2 · un Sequential de 3 couches", isinstance(essai, models.Sequential) and len(essai.layers) == 3)
verifier("Exercice 2 · flip, rotation, zoom", essai is not None and [type(c).__name__ for c in essai.layers] == ["RandomFlip", "RandomRotation", "RandomZoom"])
if essai is None:          # filet de sécurité
    def faire_augmentation():
        return models.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.1), layers.RandomZoom(0.1)], name="augmentation")
augmentation_demo = faire_augmentation()
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax in axes:
    ax.imshow(augmentation_demo(Xp_tr[:1], training=True)[0].numpy().astype("uint8")); ax.axis("off")
plt.suptitle("La même photo, 6 augmentations aléatoires"); plt.tight_layout(); plt.show()
noter("Augmentation", "miroir horizontal, rotation ±10 %, zoom ±10 %", "variantes infinies des images d'apprentissage (pas du test)")

<details><summary>Solution de l'exercice 2</summary>

```python
def faire_augmentation():
    return models.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.1), layers.RandomZoom(0.1)],
                             name="augmentation")
```
</details>

In [ ]:
pd.DataFrame(JOURNAL)

## 4. Analyse exploratoire

Cinq indicateurs pour comprendre les données avant de modéliser : à quoi ressemble « la » robe moyenne, quelles classes se ressemblent, quelles photos sont les plus variées.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for code, ax in enumerate(axes.flat):
    ax.imshow(X_train_n[y_fm_train == code].mean(axis=0)[..., 0], cmap="gray")
    ax.set_title(CLASSES_FM_FR[code]); ax.axis("off")
plt.suptitle("Indicateur 1 · image moyenne par classe"); plt.tight_layout(); plt.show()
print("Lecture : pull, manteau et chemise ont presque la même silhouette moyenne — ce sont eux que le modèle risque de confondre.")

**À toi · exercice 3 · intensité moyenne par classe**

Calcule `intensite_par_classe`, un dictionnaire `{nom de classe (fr): intensité moyenne des pixels}` sur `X_train_n` (une valeur entre 0 et 1 par classe). Quelle classe est la plus « sombre » ?

<details><summary>Indice</summary>

Une compréhension de dictionnaire : `{CLASSES_FM_FR[c]: float(X_train_n[y_fm_train == c].mean()) for c in range(10)}`.
</details>

In [ ]:
# À toi : pour chaque code de 0 à 9, la moyenne de X_train_n[y_fm_train == code]
intensite_par_classe = None

In [ ]:
verifier("Exercice 3 · 10 classes", isinstance(intensite_par_classe, dict) and len(intensite_par_classe) == 10)
verifier("Exercice 3 · valeurs entre 0 et 1", intensite_par_classe is not None and all(0 < v < 1 for v in intensite_par_classe.values()))
if intensite_par_classe is None:       # filet de sécurité
    intensite_par_classe = {CLASSES_FM_FR[code]: float(X_train_n[y_fm_train == code].mean()) for code in range(10)}
serie = pd.Series(intensite_par_classe).sort_values()
serie.plot.barh(figsize=(7, 3.5), title="Indicateur 2 · intensité moyenne des pixels par classe"); plt.tight_layout(); plt.show()
print(f"Lecture : la classe la plus sombre est « {serie.index[0]} » ({serie.iloc[0]:.2f}), la plus claire « {serie.index[-1]} » ({serie.iloc[-1]:.2f}).")

<details><summary>Solution de l'exercice 3</summary>

```python
intensite_par_classe = {CLASSES_FM_FR[code]: float(X_train_n[y_fm_train == code].mean()) for code in range(10)}
```
</details>

In [ ]:
moyennes = np.stack([X_train_n[y_fm_train == code].mean(axis=0).ravel() for code in range(10)])
corr = np.corrcoef(moyennes)
plt.figure(figsize=(6.5, 5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar(label="corrélation")
plt.xticks(range(10), CLASSES_FM_FR, rotation=45, ha="right"); plt.yticks(range(10), CLASSES_FM_FR)
plt.title("Indicateur 3 · ressemblance entre images moyennes"); plt.tight_layout(); plt.show()
paires = sorted(((corr[i, j], CLASSES_FM_FR[i], CLASSES_FM_FR[j]) for i in range(10) for j in range(i + 1, 10)), reverse=True)[:3]
print("Lecture : paires les plus ressemblantes →", ", ".join(f"{a} / {b} ({c:.2f})" for c, a, b in paires))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
pd.Series(np.bincount(y_photos), index=CLASSES_PHOTOS).plot.bar(ax=ax1, title="Indicateur 4 · photos par classe", rot=30)
couleurs_moyennes = np.stack([X_photos[y_photos == code].mean(axis=(0, 1, 2)) for code in range(len(CLASSES_PHOTOS))])
pd.DataFrame(couleurs_moyennes, index=CLASSES_PHOTOS, columns=["R", "V", "B"]).plot.bar(ax=ax2, color=["red", "green", "blue"],
                                                                                        title="Indicateur 5 · couleur moyenne", rot=30)
plt.tight_layout(); plt.show()
print("Lecture : les classes sont à peu près équilibrées ; la couleur moyenne distingue mal les classes (tournesol et pissenlit sont jaunes tous les deux) → il faut des formes, pas seulement des couleurs.")

**Ce qui oriente le choix du modèle** : les classes de Fashion-MNIST sont bien séparées par leur silhouette, sauf le trio pull / manteau / chemise → un modèle linéaire sur les pixels fera un score correct, un CNN (qui apprend des formes) fera mieux. Les photos de fleurs sont variées (fond, cadrage, lumière) et peu nombreuses → un CNN entraîné de zéro va sur-apprendre, d'où l'intérêt d'un réseau pré-entraîné sur des millions d'images.

In [ ]:
print("=== Rapport · section 4 (analyse exploratoire) ===")
print(f"Fashion-MNIST : {len(X_train_n)} images train, {len(X_test_n)} test, 10 classes équilibrées ({np.bincount(y_fm_train).min()}-{np.bincount(y_fm_train).max()} par classe)")
print(f"Paire la plus ressemblante : {paires[0][1]} / {paires[0][2]} (corrélation {paires[0][0]:.2f})")
print(f"Photos ({SOURCE_PHOTOS}) : {len(X_photos)} images {IMG_SIZE}×{IMG_SIZE}, {len(CLASSES_PHOTOS)} classes")

## 5. Modèles candidats

Toujours dans cet ordre : une **baseline « bête »** pour savoir ce qui est facile, puis des modèles de plus en plus riches. Chaque résultat va dans le tableau `RESULTATS` (modèle, jeu, accuracy test, temps). Validation : hold-out (jeu de validation pendant l'entraînement, jeu de test touché une seule fois à la fin).

- Fashion-MNIST : régression logistique sur les 784 pixels → CNN maison → CNN maison + augmentation.
- Photos : classe majoritaire → CNN maison → **MobileNetV2 gelé** (transfert).

In [ ]:
RESULTATS = []

def ajouter_resultat(modele, jeu, accuracy, temps, etape="modèles candidats"):
    RESULTATS.append({"étape": etape, "modèle": modele, "jeu": jeu, "accuracy test": round(float(accuracy), 4), "temps (s)": round(temps, 1)})
    return pd.DataFrame(RESULTATS)

In [ ]:
from sklearn.linear_model import LogisticRegression

t0 = time.time()
logreg = LogisticRegression(max_iter=300).fit(X_tr.reshape(len(X_tr), -1), y_tr)
acc_logreg = logreg.score(X_test_n.reshape(len(X_test_n), -1), y_fm_test)
ajouter_resultat("Baseline · LogReg sur pixels", "Fashion-MNIST", acc_logreg, time.time() - t0)

### CNN maison

Des blocs `Conv2D` (détecter des motifs locaux) + `MaxPooling2D` (réduire la taille), puis `Flatten` → `Dropout` (éteindre des neurones au hasard pour limiter le par-cœur) → `Dense`. `EarlyStopping` arrête l'entraînement quand la perte de validation ne baisse plus et garde les meilleurs poids.

In [ ]:
def construire_cnn(forme_entree=(28, 28, 1), n_classes=10, augmentation=None, filtres=(32, 64), dropout=0.3):
    """CNN maison : [augmentation] → blocs Conv2D/MaxPooling → Flatten → Dropout → Dense."""
    couches = [layers.Input(forme_entree)]
    if augmentation is not None:
        couches.append(augmentation)
    if forme_entree[-1] == 3:                     # photos 0-255 → 0-1
        couches.append(layers.Rescaling(1.0 / 255))
    for f in filtres:
        couches += [layers.Conv2D(f, 3, activation="relu", padding="same"), layers.MaxPooling2D()]
    couches += [layers.Flatten(), layers.Dropout(dropout), layers.Dense(64, activation="relu"),
                layers.Dense(n_classes, activation="softmax")]
    modele = models.Sequential(couches)
    modele.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modele

arret = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
cnn = construire_cnn()
cnn.summary()

In [ ]:
t0 = time.time()
hist_cnn = cnn.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=64, callbacks=[arret], verbose=2)
temps_cnn = time.time() - t0
acc_cnn = cnn.evaluate(X_test_n, y_fm_test, verbose=0)[1]
tracer_historique({"CNN": hist_cnn}, "CNN maison sur Fashion-MNIST")
ajouter_resultat("CNN maison", "Fashion-MNIST", acc_cnn, temps_cnn)

**À toi · exercice 4 · un CNN plus profond**

Avec les arguments de `construire_cnn()`, crée `cnn_v2` : **3 blocs** de filtres `(32, 64, 128)` et un `dropout` de `0.5`. Plus de blocs = des motifs plus abstraits ; plus de dropout = moins de par-cœur.

<details><summary>Indice</summary>

`construire_cnn(filtres=(32, 64, 128), dropout=0.5)` — les autres arguments gardent leur valeur par défaut.
</details>

In [ ]:
# À toi : construire_cnn(...) avec filtres=(32, 64, 128) et dropout=0.5
cnn_v2 = None

In [ ]:
nb_conv = 0 if cnn_v2 is None else sum(isinstance(c, layers.Conv2D) for c in cnn_v2.layers)
verifier("Exercice 4 · 3 couches Conv2D", nb_conv == 3)
verifier("Exercice 4 · dropout 0.5 et 10 sorties", cnn_v2 is not None and any(isinstance(c, layers.Dropout) and c.rate == 0.5 for c in cnn_v2.layers) and cnn_v2.layers[-1].units == 10)
if cnn_v2 is None:         # filet de sécurité
    cnn_v2 = construire_cnn(filtres=(32, 64, 128), dropout=0.5)

<details><summary>Solution de l'exercice 4</summary>

```python
cnn_v2 = construire_cnn(filtres=(32, 64, 128), dropout=0.5)
```
</details>

In [ ]:
t0 = time.time()
hist_v2 = cnn_v2.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=64, callbacks=[arret], verbose=2)
acc_v2 = cnn_v2.evaluate(X_test_n, y_fm_test, verbose=0)[1]
ajouter_resultat("CNN maison v2 (3 blocs, dropout 0.5)", "Fashion-MNIST", acc_v2, time.time() - t0)

In [ ]:
cnn_aug = construire_cnn(augmentation=faire_augmentation())
t0 = time.time()
hist_aug = cnn_aug.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=64, callbacks=[arret], verbose=2)
temps_aug = time.time() - t0
acc_aug = cnn_aug.evaluate(X_test_n, y_fm_test, verbose=0)[1]
tracer_historique({"sans": hist_cnn, "avec augmentation": hist_aug}, "Effet de l'augmentation (Fashion-MNIST)")
ajouter_resultat("CNN maison + augmentation", "Fashion-MNIST", acc_aug, temps_aug)

Lecture : avec l'augmentation, la courbe d'apprentissage est plus basse (la tâche est plus dure) et l'écart train / validation se resserre. Sur 2 epochs elle ne fait pas encore gagner de points de test ; c'est sur un entraînement long (`MODE_RAPIDE = False`) qu'elle empêche le sur-apprentissage.

### Sur les photos : CNN maison, puis transfert

Le même CNN maison sur les photos donne la référence. Puis **MobileNetV2** (2,3 millions de paramètres, pré-entraîné sur ImageNet — 1,3 million d'images, 1 000 classes) : on **gèle** toutes ses couches (`trainable = False`) et on n'entraîne que la petite tête de classification que l'on ajoute.

In [ ]:
acc_majoritaire = np.bincount(yp_test).max() / len(yp_test)
ajouter_resultat("Baseline · classe majoritaire", SOURCE_PHOTOS, acc_majoritaire, 0)

cnn_photos = construire_cnn((IMG_SIZE, IMG_SIZE, 3), len(CLASSES_PHOTOS), augmentation=faire_augmentation(), filtres=(32, 64, 128))
t0 = time.time()
hist_cnn_photos = cnn_photos.fit(Xp_tr, yp_tr, validation_data=(Xp_val, yp_val), epochs=EPOCHS, batch_size=32, callbacks=[arret], verbose=2)
acc_cnn_photos = cnn_photos.evaluate(Xp_test, yp_test, verbose=0)[1]
ajouter_resultat("CNN maison", SOURCE_PHOTOS, acc_cnn_photos, time.time() - t0)

In [ ]:
def construire_transfert(n_classes, augmentation=None):
    """MobileNetV2 gelé (pré-entraîné ImageNet) + tête de classification entraînable. Renvoie (modèle, base)."""
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False                                 # gelé : seules les couches ajoutées apprennent
    entree = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(entree) if augmentation is not None else entree
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)        # MobileNetV2 attend des pixels entre -1 et 1
    x = base(x, training=False)                            # training=False fige aussi les statistiques BatchNorm
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    sortie = layers.Dense(n_classes, activation="softmax")(x)
    modele = models.Model(entree, sortie)
    modele.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modele, base

transfert, base = construire_transfert(len(CLASSES_PHOTOS), faire_augmentation())
nb_entrainables = sum(int(np.prod(w.shape)) for w in transfert.trainable_weights)
print(f"Paramètres : {transfert.count_params():,} · entraînables : {nb_entrainables:,} ({nb_entrainables / transfert.count_params():.1%})")

In [ ]:
t0 = time.time()
hist_gel = transfert.fit(Xp_tr, yp_tr, validation_data=(Xp_val, yp_val), epochs=EPOCHS, batch_size=32, callbacks=[arret], verbose=2)
temps_gel = time.time() - t0
acc_gel = transfert.evaluate(Xp_test, yp_test, verbose=0)[1]
tracer_historique({"CNN maison": hist_cnn_photos, "MobileNetV2 gelé": hist_gel}, f"Photos ({SOURCE_PHOTOS}) : de zéro contre transfert")
ajouter_resultat("MobileNetV2 gelé (transfert)", SOURCE_PHOTOS, acc_gel, temps_gel)

In [ ]:
print("=== Rapport · section 5 (modèles candidats) ===")
print(pd.DataFrame(RESULTATS).to_string(index=False))

## 6. Tuning / fine-tuning

Le transfert « gelé » réutilise les motifs d'ImageNet tels quels. Le **fine-tuning** va un cran plus loin : on **dégèle les dernières couches** de MobileNetV2 (celles qui détectent les motifs les plus abstraits, donc les plus spécifiques à ImageNet) et on les ré-entraîne **doucement** — learning rate 100 fois plus petit — pour les adapter aux fleurs sans détruire ce qui a été appris. Deux réglages à choisir : combien de couches dégeler (`NB_DEGEL`) et le learning rate.

Règle d'or : ne dégeler qu'**après** que la tête a convergé (sinon ses gradients aléatoires abîment les poids pré-entraînés), et rester à un learning rate très faible.

**À toi · exercice 5 · dégeler les 30 dernières couches**

`base.trainable = True` rend tout MobileNetV2 entraînable. Regèle toutes les couches **sauf les `NB_DEGEL` dernières** : boucle sur `base.layers[:-NB_DEGEL]` et mets leur attribut `trainable` à `False`.

<details><summary>Indice</summary>

`for couche in base.layers[:-NB_DEGEL]:` puis `couche.trainable = False` (indenté).
</details>

In [ ]:
NB_DEGEL = 30
base.trainable = True
# À toi : boucle sur base.layers[:-NB_DEGEL] → couche.trainable = False

nb_couches_entrainables = sum(couche.trainable for couche in base.layers)
print("Couches entraînables dans MobileNetV2 :", nb_couches_entrainables, "/", len(base.layers))

In [ ]:
verifier("Exercice 5 · exactement 30 couches dégelées", nb_couches_entrainables == NB_DEGEL)
if nb_couches_entrainables != NB_DEGEL:   # filet de sécurité
    for couche in base.layers[:-NB_DEGEL]:
        couche.trainable = False

<details><summary>Solution de l'exercice 5</summary>

```python
NB_DEGEL = 30
base.trainable = True
for couche in base.layers[:-NB_DEGEL]:
    couche.trainable = False

nb_couches_entrainables = sum(couche.trainable for couche in base.layers)
print("Couches entraînables dans MobileNetV2 :", nb_couches_entrainables, "/", len(base.layers))
```
</details>

**À toi · exercice 6 · recompiler avec un learning rate minuscule**

Après avoir changé ce qui est entraînable, il faut **recompiler**. Fixe `LR_FINE_TUNING = 1e-5` et recompile `transfert` avec `keras.optimizers.Adam(LR_FINE_TUNING)`, la même loss (`sparse_categorical_crossentropy`) et la métrique `accuracy`.

<details><summary>Indice</summary>

Exactement comme dans `construire_transfert()`, avec `Adam(LR_FINE_TUNING)` à la place de `Adam(1e-3)`.
</details>

In [ ]:
# À toi : LR_FINE_TUNING = 1e-5 puis transfert.compile(...)
LR_FINE_TUNING = None

In [ ]:
lr_actuel = float(transfert.optimizer.learning_rate)
verifier("Exercice 6 · learning rate 1e-5", LR_FINE_TUNING is not None and proche(lr_actuel, 1e-5, 1e-7))
if LR_FINE_TUNING is None:                # filet de sécurité
    LR_FINE_TUNING = 1e-5
    transfert.compile(optimizer=keras.optimizers.Adam(LR_FINE_TUNING), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
nb_entrainables_ft = sum(int(np.prod(w.shape)) for w in transfert.trainable_weights)
print(f"Paramètres entraînables : {nb_entrainables:,} (gelé) → {nb_entrainables_ft:,} (fine-tuning) · learning rate {float(transfert.optimizer.learning_rate):.0e}")

<details><summary>Solution de l'exercice 6</summary>

```python
LR_FINE_TUNING = 1e-5
transfert.compile(optimizer=keras.optimizers.Adam(LR_FINE_TUNING), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
```
</details>

In [ ]:
t0 = time.time()
hist_ft = transfert.fit(Xp_tr, yp_tr, validation_data=(Xp_val, yp_val), epochs=EPOCHS, batch_size=32, callbacks=[arret], verbose=2)
temps_ft = time.time() - t0
acc_ft = transfert.evaluate(Xp_test, yp_test, verbose=0)[1]
ajouter_resultat(f"MobileNetV2 fine-tuné ({NB_DEGEL} couches, LR {LR_FINE_TUNING:.0e})", SOURCE_PHOTOS, acc_ft, temps_ft, etape="fine-tuning")

In [ ]:
acc_val = hist_gel.history["val_accuracy"] + hist_ft.history["val_accuracy"]
loss_val = hist_gel.history["val_loss"] + hist_ft.history["val_loss"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(loss_val, marker="o"); ax1.axvline(len(hist_gel.history["val_loss"]) - 0.5, color="gray", ls="--"); ax1.set_title("Perte de validation")
ax2.plot(acc_val, marker="o"); ax2.axvline(len(hist_gel.history["val_accuracy"]) - 0.5, color="gray", ls="--"); ax2.set_title("Accuracy de validation")
for ax in (ax1, ax2):
    ax.set_xlabel("epoch"); ax.text(0.02, 0.9, "gelé", transform=ax.transAxes); ax.text(0.7, 0.9, "fine-tuning", transform=ax.transAxes)
plt.suptitle("Avant / après le dégel"); plt.tight_layout(); plt.show()
print(f"Accuracy test : {acc_gel:.3f} (gelé) → {acc_ft:.3f} (fine-tuné)")

In [ ]:
print("=== Rapport · section 6 (fine-tuning) ===")
print(f"Couches dégelées : {NB_DEGEL} / {len(base.layers)} · learning rate : {LR_FINE_TUNING:.0e} · epochs : {EPOCHS}")
print(f"Accuracy test avant / après : {acc_gel:.3f} → {acc_ft:.3f} ({(acc_ft - acc_gel) * 100:+.1f} points)")
print(pd.DataFrame(RESULTATS).to_string(index=False))

## 7. Interprétation

Un score global ne suffit pas : **où** le modèle se trompe-t-il, et **que regarde-t-il** pour décider ? Trois outils : la matrice de confusion, les erreurs typiques affichées, et Grad-CAM.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

proba_test = transfert.predict(Xp_test, verbose=0)
pred_test = proba_test.argmax(axis=1)
cm = confusion_matrix(yp_test, pred_test)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES_PHOTOS).plot(ax=ax1, cmap="Blues", xticks_rotation=45, colorbar=False)
ax1.set_title(f"Photos ({SOURCE_PHOTOS}) · MobileNetV2 fine-tuné")
pred_fm = cnn.predict(X_test_n, verbose=0).argmax(axis=1)
ConfusionMatrixDisplay(confusion_matrix(y_fm_test, pred_fm), display_labels=CLASSES_FM_FR).plot(ax=ax2, cmap="Blues", xticks_rotation=45, colorbar=False)
ax2.set_title("Fashion-MNIST · CNN maison")
plt.tight_layout(); plt.show()

**À toi · exercice 7 · accuracy par classe**

À partir de la matrice `cm` (lignes = vraies classes, colonnes = prédictions), calcule `acc_par_classe` : un tableau numpy avec, pour chaque classe, la part de ses images correctement classées (diagonale / total de la ligne). Puis `pire_classe`, le nom de la classe la moins bien reconnue.

<details><summary>Indice</summary>

`cm.diagonal()` donne les bonnes réponses par classe, `cm.sum(axis=1)` le nombre d'images par vraie classe.
</details>

In [ ]:
# À toi : cm.diagonal() / cm.sum(axis=1), puis CLASSES_PHOTOS[...argmin()]
acc_par_classe = None
pire_classe = None

In [ ]:
verifier("Exercice 7 · une valeur par classe", acc_par_classe is not None and len(acc_par_classe) == len(CLASSES_PHOTOS))
verifier("Exercice 7 · cohérent avec l'accuracy globale", acc_par_classe is not None and proche((acc_par_classe * cm.sum(axis=1)).sum() / cm.sum(), acc_ft, 1e-3))
verifier("Exercice 7 · pire classe", pire_classe is not None and pire_classe == CLASSES_PHOTOS[np.argmin(cm.diagonal() / cm.sum(axis=1))])
if acc_par_classe is None:                # filet de sécurité
    acc_par_classe = cm.diagonal() / cm.sum(axis=1)
    pire_classe = CLASSES_PHOTOS[acc_par_classe.argmin()]
print(pd.Series(acc_par_classe, index=CLASSES_PHOTOS).round(3).sort_values().to_string())

<details><summary>Solution de l'exercice 7</summary>

```python
acc_par_classe = cm.diagonal() / cm.sum(axis=1)
pire_classe = CLASSES_PHOTOS[acc_par_classe.argmin()]
```
</details>

In [ ]:
erreurs = np.where(pred_test != yp_test)[0]
print(f"{len(erreurs)} erreurs sur {len(yp_test)} images de test ({len(erreurs) / len(yp_test):.1%})")
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.flat, erreurs[:8]):
    ax.imshow(Xp_test[i].astype("uint8"))
    ax.set_title(f"vrai : {CLASSES_PHOTOS[yp_test[i]]}\nprédit : {CLASSES_PHOTOS[pred_test[i]]} ({proba_test[i].max():.0%})", fontsize=9)
    ax.axis("off")
plt.suptitle("Erreurs typiques"); plt.tight_layout(); plt.show()

### Grad-CAM : où regarde le modèle ?

Grad-CAM prend la dernière carte de caractéristiques de MobileNetV2 (`out_relu`, une petite grille de 1 280 canaux), calcule le **gradient** de la probabilité prédite par rapport à cette carte, et pondère chaque canal par l'importance de son gradient. On obtient une carte de chaleur : les zones rouges sont celles qui ont poussé le modèle vers sa décision. Si le modèle regarde le fond plutôt que la fleur, on le voit tout de suite.

In [ ]:
extracteur = models.Model(base.input, base.get_layer("out_relu").output)   # image → cartes de caractéristiques
tete = models.Sequential(transfert.layers[-3:])                              # GlobalAveragePooling → Dropout → Dense

def grad_cam(image):
    """Carte de chaleur Grad-CAM (0-1) d'une image (H, W, 3) en 0-255 ; renvoie (carte, classe prédite, confiance)."""
    x = tf.constant(image[np.newaxis] / 127.5 - 1, dtype=tf.float32)
    with tf.GradientTape() as tape:
        cartes = extracteur(x)
        tape.watch(cartes)
        probas = tete(cartes, training=False)
        score = probas[0, tf.argmax(probas[0])]
    gradients = tape.gradient(score, cartes)
    poids = tf.reduce_mean(gradients, axis=(0, 1, 2))          # importance de chaque canal
    carte = tf.reduce_sum(cartes[0] * poids, axis=-1)
    carte = tf.maximum(carte, 0) / (tf.reduce_max(carte) + 1e-8)
    return carte.numpy(), int(tf.argmax(probas[0])), float(tf.reduce_max(probas[0]))

def afficher_grad_cam(image, ax, titre=""):
    carte, classe, confiance = grad_cam(image)
    ax.imshow(image.astype("uint8"))
    ax.imshow(carte, cmap="jet", alpha=0.45, extent=(0, IMG_SIZE, IMG_SIZE, 0), interpolation="bilinear")
    ax.set_title(f"{titre}{CLASSES_PHOTOS[classe]} ({confiance:.0%})", fontsize=9); ax.axis("off")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, i in zip(axes, np.where(pred_test == yp_test)[0][:3]):
    afficher_grad_cam(Xp_test[i], ax, "prédit : ")
plt.suptitle("Grad-CAM sur 3 images bien classées"); plt.tight_layout(); plt.show()

**À toi · exercice 8 · Grad-CAM sur une erreur**

Prends la première image mal classée (`idx_erreur`) et calcule `carte, classe, confiance = grad_cam(...)`. Affiche-la avec `afficher_grad_cam`. Le modèle regardait-il la fleur ou le fond ?

<details><summary>Indice</summary>

Trois valeurs sont renvoyées par `grad_cam` ; `afficher_grad_cam(image, ax, titre)` a besoin d'un axe matplotlib (`fig, ax = plt.subplots()`).
</details>

In [ ]:
idx_erreur = erreurs[0]
# À toi : carte, classe, confiance = grad_cam(Xp_test[idx_erreur])
carte = None

In [ ]:
verifier("Exercice 8 · une carte 2D normalisée", carte is not None and carte.ndim == 2 and proche(carte.max(), 1.0, 1e-3))

<details><summary>Solution de l'exercice 8</summary>

```python
idx_erreur = erreurs[0]
carte, classe, confiance = grad_cam(Xp_test[idx_erreur])
fig, ax = plt.subplots(figsize=(3.5, 3.5))
afficher_grad_cam(Xp_test[idx_erreur], ax, f"vrai : {CLASSES_PHOTOS[yp_test[idx_erreur]]} · prédit : ")
plt.show()
```
</details>

### Prédire une photo prise sur le web

Colle l'URL d'une photo (une fleur si `tf_flowers` est chargé). Le modèle attend des pixels 0-255 : aucun prétraitement à faire, la mise à l'échelle est dans le modèle.

In [ ]:
import requests
from PIL import Image

URL_IMAGE = "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"   # ← remplace par ta photo

try:
    reponse = requests.get(URL_IMAGE, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
    reponse.raise_for_status()
    image_web = Image.open(io.BytesIO(reponse.content)).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    probas_web = transfert.predict(np.array(image_web, dtype="float32")[np.newaxis], verbose=0)[0]
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    afficher_grad_cam(np.array(image_web, dtype="float32"), ax, "prédit : ")
    plt.show()
    print(pd.Series(probas_web, index=CLASSES_PHOTOS).sort_values(ascending=False).round(3).to_string())
except Exception as e:
    print("Image inaccessible (", type(e).__name__, ") : vérifie l'URL ou ta connexion.")

**Trois enseignements**
1. Sur Fashion-MNIST, les erreurs se concentrent sur le trio pull / manteau / chemise, exactement les classes que l'image moyenne annonçait proches : la matrice de confusion confirme l'EDA.
2. Sur les photos, le transfert bat le CNN maison dès la première epoch, et le fine-tuning ajoute encore quelques points quand on lui laisse le temps (`MODE_RAPIDE = False`).
3. Grad-CAM montre si la décision repose sur la fleur ou sur le fond (herbe, ciel) : c'est le premier test à faire avant de déployer un modèle d'images.

In [ ]:
print("=== Rapport · section 7 (interprétation) ===")
print(f"Erreurs test ({SOURCE_PHOTOS}) : {len(erreurs)} / {len(yp_test)} · pire classe : {pire_classe} ({acc_par_classe.min():.1%})")
i, j = np.unravel_index(np.argmax(cm - np.diag(cm.diagonal())), cm.shape)
print(f"Confusion la plus fréquente : {CLASSES_PHOTOS[i]} prédit {CLASSES_PHOTOS[j]} ({cm[i, j]} fois)")
print(f"Fashion-MNIST · accuracy CNN maison : {acc_cnn:.3f} · LogReg : {acc_logreg:.3f}")

## 8. Conclusion et recommandation

**Réponse à la question métier** : oui, la reconnaissance automatique est exploitable. Sur les vêtements, un CNN maison dépasse nettement le modèle linéaire ; sur les photos de fleurs, entraîner de zéro ne marche pas avec si peu d'images, mais un réseau pré-entraîné adapté par transfert puis fine-tuning atteint un niveau utilisable. **Chiffre clé** : l'accuracy test du modèle fine-tuné (tableau ci-dessus), à comparer à la classe majoritaire (≈ 20 %).

**Recommandation** : déployer le modèle fine-tuné avec un seuil de confiance (par ex. 80 %) ; en dessous, la photo part en relecture humaine. Suivre la matrice de confusion en production.

**Limites** : en `MODE_RAPIDE`, 2 epochs et 15 % des photos — les scores sont volontairement sous-estimés ; les photos de test viennent de la même collection que l'apprentissage (photos de téléphone floues ou de nuit : à tester) ; 5 espèces seulement.

**Avec plus de temps** : version complète sur GPU, essayer `NB_DEGEL` 10 / 60 / 100 et un modèle plus gros (EfficientNet), ajouter une classe « autre », collecter des photos réelles d'utilisateurs.

In [ ]:
# Ma synthèse (3-5 lignes, à recopier en dernière slide) : le résultat, le chiffre clé, ce que je recommande
MA_SYNTHESE = """
"""
print(MA_SYNTHESE.strip() or "→ écris ta synthèse ci-dessus")

In [ ]:
tableau = pd.DataFrame(RESULTATS)
verifier("Tableau comparatif : au moins 6 modèles", len(tableau) >= 6)
verifier("Le transfert bat la classe majoritaire", acc_gel > acc_majoritaire)
verifier("Le CNN maison bat la baseline LogReg sur Fashion-MNIST", acc_cnn > acc_logreg)
verifier("Fine-tuning mesuré avant / après", "fine-tuning" in set(tableau["étape"]))
verifier("Ma synthèse est écrite", len(MA_SYNTHESE.strip()) > 50)
tableau

## 9. Pour aller plus loin

- **Régler le fine-tuning** : relance la section 6 avec `NB_DEGEL = 10` puis `100`, et un learning rate de `1e-4` — note chaque essai dans `RESULTATS`. Guide Keras : https://keras.io/guides/transfer_learning/
- **Changer de squelette** : `keras.applications.EfficientNetB0` ou `ResNet50V2` à la place de MobileNetV2 (même code, `preprocess_input` différent) : https://keras.io/api/applications/
- **Grad-CAM en détail** : l'exemple officiel Keras https://keras.io/examples/vision/grad_cam/ et l'article d'origine https://arxiv.org/abs/1610.02391
- **Visualiser les convolutions** : https://poloclub.github.io/cnn-explainer/ (interactif) et https://distill.pub/2017/feature-visualization/
- **Un autre dataset** : les 102 espèces de fleurs d'Oxford (`oxford_flowers102` dans TensorFlow Datasets) ou tes propres photos rangées en dossiers, chargées avec `keras.utils.image_dataset_from_directory`.